# Part 9 · GCP Libraries + boto3 (AWS)
> google-cloud-storage / bigquery / pubsub / secret-manager / boto3 S3

## 1. 认证

In [ ]:
# --- GCP 认证方式 ---

# 方式 1：ADC (Application Default Credentials) - 推荐
# 本地开发：gcloud auth application-default login
# GCE/Cloud Run/GKE：自动使用实例 Service Account
# 代码无需任何改变，会自动找凭证

# 方式 2：Service Account JSON 文件
import os
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '/path/to/key.json'

# 方式 3：显式加载 credentials
from google.oauth2 import service_account
creds = service_account.Credentials.from_service_account_file(
    '/path/to/key.json',
    scopes=['https://www.googleapis.com/auth/cloud-platform']
)
# 传给 client
from google.cloud import storage
client = storage.Client(credentials=creds, project='my-project')

# 方式 4：从 Secret Manager 读取 JSON
# （见下方 Secret Manager 示例）

# 检查当前 ADC
import google.auth
creds, project = google.auth.default()
print(f'Project: {project}')

## 2. Cloud Storage (GCS)

In [ ]:
from google.cloud import storage
import io
import pandas as pd

client = storage.Client(project='my-project')

# --- Bucket 操作 ---
bucket = client.bucket('my-bucket')         # 获取 bucket 引用（不验证存在）
bucket = client.get_bucket('my-bucket')     # 获取 bucket（会验证，需要权限）
client.create_bucket('new-bucket', location='US')
client.list_buckets()                        # 列出所有 bucket

# --- Blob（文件）操作 ---
blob = bucket.blob('data/orders/2024-01.csv')   # 获取引用

# 上传
blob.upload_from_filename('/local/data.csv')                # 从文件
blob.upload_from_string('hello,world\n', content_type='text/csv')  # 从字符串
blob.upload_from_file(open('data.csv', 'rb'))               # 从文件对象

# DataFrame → GCS（内存方式，不写本地文件）
buf = io.BytesIO()
df.to_parquet(buf, index=False)
buf.seek(0)
bucket.blob('data/orders.parquet').upload_from_file(buf, content_type='application/octet-stream')

# 下载
blob.download_to_filename('/local/data.csv')    # 到文件
content = blob.download_as_bytes()              # 到 bytes
text    = blob.download_as_text(encoding='utf-8')  # 到字符串

# GCS → DataFrame（内存）
blob = bucket.blob('data/orders.parquet')
buf = io.BytesIO(blob.download_as_bytes())
df = pd.read_parquet(buf)

# pandas 直接读 GCS（需要 gcsfs）
df = pd.read_csv('gs://my-bucket/data/orders.csv')
df = pd.read_parquet('gs://my-bucket/data/orders.parquet')
df.to_parquet('gs://my-bucket/output/result.parquet', index=False)

# --- 列出文件 ---
blobs = client.list_blobs('my-bucket', prefix='data/orders/', delimiter='/')
for blob in blobs:
    print(blob.name, blob.size, blob.updated)

# 只列出子目录
blobs = client.list_blobs('my-bucket', prefix='data/', delimiter='/')
for prefix in blobs.prefixes:  # 虚拟目录
    print(prefix)

# --- 元数据 & 属性 ---
blob = bucket.get_blob('data/orders.csv')   # None if not exists
blob.name             # 'data/orders.csv'
blob.size             # 字节数
blob.content_type     # 'text/csv'
blob.updated          # datetime
blob.md5_hash         # MD5 校验
blob.exists()         # 是否存在

# --- 删除 ---
blob.delete()
bucket.delete_blobs(blobs=list(client.list_blobs('my-bucket', prefix='tmp/')))  # 批量删除

# --- 复制 & 移动 ---
src_blob = bucket.blob('data/old.csv')
dst_blob = bucket.copy_blob(src_blob, bucket, 'data/new.csv')  # 同 bucket 复制
src_blob.delete()  # 删除源文件 = 移动

# --- 生成签名 URL（临时访问，无需认证）---
from datetime import timedelta
url = blob.generate_signed_url(
    expiration=timedelta(hours=1),
    method='GET'
)

# --- 访问控制 ---
blob.make_public()                # 公开读
blob.acl.reload()
blob.acl.all().grant_read()       # 所有人可读
blob.acl.save()

## 3. BigQuery

In [ ]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project='my-project')

# --- 查询 ---
# 简单查询
df = client.query('SELECT * FROM `project.dataset.orders` LIMIT 100').to_dataframe()

# 参数化查询（安全）
query = '''
    SELECT customer_id, SUM(total) AS total_spent
    FROM `project.dataset.orders`
    WHERE DATE(created_at) BETWEEN @start AND @end
    GROUP BY 1
'''
job_config = bigquery.QueryJobConfig(query_parameters=[
    bigquery.ScalarQueryParameter('start', 'DATE', '2024-01-01'),
    bigquery.ScalarQueryParameter('end',   'DATE', '2024-01-31'),
])
df = client.query(query, job_config=job_config).to_dataframe()

# 查询到大 DataFrame（带进度）
job = client.query(query, job_config=job_config)
job.result()                         # 等待完成
df = job.to_dataframe()
print(f'Rows: {job.total_rows}, Bytes: {job.total_bytes_processed}')

# --- 写入 DataFrame → BQ ---
job = client.load_table_from_dataframe(
    df,
    'project.dataset.table',
    job_config=bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,   # 覆盖
        # 或 WRITE_APPEND, WRITE_EMPTY
        autodetect=True,    # 自动推断 schema
        schema=[            # 或手动指定
            bigquery.SchemaField('id', 'INTEGER'),
            bigquery.SchemaField('name', 'STRING'),
            bigquery.SchemaField('created_at', 'TIMESTAMP'),
        ]
    )
)
job.result()    # 等待

# --- 从 GCS 加载 ---
job = client.load_table_from_uri(
    'gs://my-bucket/data/orders/*.parquet',
    'project.dataset.orders',
    job_config=bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.PARQUET,
        write_disposition='WRITE_TRUNCATE',
    )
)
job.result()

# --- 导出到 GCS ---
job = client.extract_table(
    'project.dataset.orders',
    'gs://my-bucket/export/orders_*.csv',
    job_config=bigquery.ExtractJobConfig(
        destination_format='CSV',
        compression='GZIP',
        print_header=True,
    )
)
job.result()

# --- 表操作 ---
# 获取表
table = client.get_table('project.dataset.orders')
table.schema           # 字段列表
table.num_rows         # 行数
table.num_bytes        # 大小
table.modified         # 修改时间
table.partitioning_type  # 分区类型

# 创建表
schema = [
    bigquery.SchemaField('id', 'INTEGER', mode='REQUIRED'),
    bigquery.SchemaField('name', 'STRING'),
    bigquery.SchemaField('created_at', 'TIMESTAMP'),
    bigquery.SchemaField('tags', 'STRING', mode='REPEATED'),  # 数组
]
table = bigquery.Table('project.dataset.new_table', schema=schema)
table.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY,
    field='created_at'
)
table.clustering_fields = ['customer_id', 'status']
client.create_table(table, exists_ok=True)

# 删除表
client.delete_table('project.dataset.old_table', not_found_ok=True)

# 列出 dataset 下所有表
for t in client.list_tables('project.dataset'):
    print(t.table_id, t.table_type)

## 4. Pub/Sub

In [ ]:
from google.cloud import pubsub_v1
import json

# --- 发布 ---
publisher = pubsub_v1.PublisherClient()
topic_path = publisher.topic_path('my-project', 'my-topic')

# 发单条
data = json.dumps({'event': 'order_created', 'order_id': 123}).encode('utf-8')
future = publisher.publish(topic_path, data,                  # 数据必须是 bytes
                           origin='python', username='user')  # 可选属性
msg_id = future.result()   # 阻塞等待确认

# 批量发布
batch_settings = pubsub_v1.types.BatchSettings(
    max_messages=100,        # 每批最多 100 条
    max_bytes=1024*1024,     # 每批最大 1MB
    max_latency=0.05,        # 最长等待 50ms
)
publisher = pubsub_v1.PublisherClient(batch_settings=batch_settings)

futures = [publisher.publish(topic_path, json.dumps(msg).encode()) for msg in messages]
results = [f.result() for f in futures]   # 等所有发完

# --- 订阅（拉取模式）---
subscriber = pubsub_v1.SubscriberClient()
sub_path = subscriber.subscription_path('my-project', 'my-subscription')

# 同步拉取（适合批量处理）
response = subscriber.pull(request={'subscription': sub_path, 'max_messages': 100})
ack_ids = []
for msg in response.received_messages:
    data = json.loads(msg.message.data.decode())
    attrs = dict(msg.message.attributes)
    print(data, attrs)
    ack_ids.append(msg.ack_id)

if ack_ids:
    subscriber.acknowledge(request={'subscription': sub_path, 'ack_ids': ack_ids})

# 异步流式拉取
def callback(message):
    data = json.loads(message.data.decode())
    print(data)
    message.ack()   # 确认（不 ack 则超时重发）
    # message.nack()  # 不确认（立刻重发）

streaming_pull = subscriber.subscribe(sub_path, callback=callback)
with subscriber:
    try:
        streaming_pull.result(timeout=30)   # 30 秒后停止
    except Exception:
        streaming_pull.cancel()

## 5. Secret Manager

In [ ]:
from google.cloud import secretmanager

client = secretmanager.SecretManagerServiceClient()

def get_secret(secret_id, project='my-project', version='latest'):
    """从 Secret Manager 读取 secret 值"""
    name = f'projects/{project}/secrets/{secret_id}/versions/{version}'
    response = client.access_secret_version(request={'name': name})
    return response.payload.data.decode('utf-8')

# 使用
db_password = get_secret('db-password')
api_key     = get_secret('stripe-api-key', version='3')  # 指定版本

# 读取 JSON secret（如 Service Account Key）
import json
sa_info = json.loads(get_secret('my-sa-key'))
from google.oauth2 import service_account
creds = service_account.Credentials.from_service_account_info(sa_info)

# 创建 secret
parent = f'projects/my-project'
secret = client.create_secret(request={
    'parent': parent,
    'secret_id': 'new-secret',
    'secret': {'replication': {'automatic': {}}}
})
client.add_secret_version(request={
    'parent': secret.name,
    'payload': {'data': b'secret-value'}
})

## 6. boto3 — AWS S3（及其他 AWS 服务）

In [ ]:
import boto3
import io
import pandas as pd

# --- 认证方式（按优先级自动查找）---
# 1. 环境变量: AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_SESSION_TOKEN
# 2. ~/.aws/credentials 文件
# 3. IAM Role（EC2/Lambda/ECS）

# 显式指定
session = boto3.Session(
    aws_access_key_id='ACCESS_KEY',
    aws_secret_access_key='SECRET_KEY',
    region_name='us-east-1'
)
s3 = session.client('s3')

# 默认认证
s3 = boto3.client('s3', region_name='us-east-1')
s3_resource = boto3.resource('s3')   # 高级 ORM 风格

# --- S3 Client 操作 ---
# 上传
s3.upload_file('/local/data.csv', 'my-bucket', 'data/orders.csv')  # 文件路径
s3.upload_fileobj(open('data.csv','rb'), 'my-bucket', 'data/orders.csv')  # 文件对象

# DataFrame → S3（内存）
buf = io.BytesIO()
df.to_parquet(buf, index=False)
buf.seek(0)
s3.upload_fileobj(buf, 'my-bucket', 'data/orders.parquet')

# 下载
s3.download_file('my-bucket', 'data/orders.csv', '/local/data.csv')
buf = io.BytesIO()
s3.download_fileobj('my-bucket', 'data/orders.parquet', buf)
buf.seek(0)
df = pd.read_parquet(buf)

# 读取内容（不写文件）
obj = s3.get_object(Bucket='my-bucket', Key='data/orders.csv')
content = obj['Body'].read().decode('utf-8')
df = pd.read_csv(io.StringIO(content))

# pandas 直接读 S3（需要 s3fs）
df = pd.read_parquet('s3://my-bucket/data/orders.parquet')
df = pd.read_csv('s3://my-bucket/data/orders.csv')
df.to_parquet('s3://my-bucket/output/result.parquet', index=False)

# 列出文件
response = s3.list_objects_v2(Bucket='my-bucket', Prefix='data/')
for obj in response.get('Contents', []):
    print(obj['Key'], obj['Size'], obj['LastModified'])

# 分页（超过 1000 个对象）
paginator = s3.get_paginator('list_objects_v2')
pages = paginator.paginate(Bucket='my-bucket', Prefix='data/')
all_keys = [obj['Key'] for page in pages for obj in page.get('Contents', [])]

# 删除
s3.delete_object(Bucket='my-bucket', Key='data/old.csv')
s3.delete_objects(Bucket='my-bucket', Delete={
    'Objects': [{'Key': k} for k in keys_to_delete]
})

# 复制
s3.copy({'Bucket': 'src-bucket', 'Key': 'src/file.csv'}, 'dst-bucket', 'dst/file.csv')

# 生成预签名 URL（临时访问）
url = s3.generate_presigned_url(
    'get_object',
    Params={'Bucket': 'my-bucket', 'Key': 'data/report.pdf'},
    ExpiresIn=3600    # 1小时有效
)

# 检查文件是否存在
import botocore
def s3_exists(bucket, key):
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except botocore.exceptions.ClientError as e:
        if e.response['Error']['Code'] == '404':
            return False
        raise

# --- S3 Resource（ORM 风格，更简洁）---
s3r = boto3.resource('s3')
bucket = s3r.Bucket('my-bucket')
bucket.upload_file('/local/data.csv', 'data/orders.csv')
bucket.download_file('data/orders.csv', '/local/data.csv')
for obj in bucket.objects.filter(Prefix='data/'):
    print(obj.key)

## 7. 常用工具函数

In [ ]:
# --- GCS 工具函数 ---
from google.cloud import storage
from pathlib import Path
import io, json
import pandas as pd

def gcs_read_parquet(bucket_name, blob_path, project=None):
    """从 GCS 读取 Parquet 为 DataFrame"""
    client = storage.Client(project=project)
    blob = client.bucket(bucket_name).blob(blob_path)
    return pd.read_parquet(io.BytesIO(blob.download_as_bytes()))

def gcs_write_parquet(df, bucket_name, blob_path, project=None):
    """将 DataFrame 写入 GCS Parquet"""
    client = storage.Client(project=project)
    buf = io.BytesIO()
    df.to_parquet(buf, index=False)
    buf.seek(0)
    client.bucket(bucket_name).blob(blob_path).upload_from_file(buf)

def gcs_list_files(bucket_name, prefix='', suffix='.parquet', project=None):
    """列出 GCS 中匹配的文件"""
    client = storage.Client(project=project)
    return [
        b.name for b in client.list_blobs(bucket_name, prefix=prefix)
        if b.name.endswith(suffix)
    ]

def gcs_read_json(bucket_name, blob_path, project=None):
    client = storage.Client(project=project)
    return json.loads(client.bucket(bucket_name).blob(blob_path).download_as_text())

# --- S3 工具函数 ---
import boto3, io
import pandas as pd

def s3_read_parquet(bucket, key, **kwargs):
    s3 = boto3.client('s3')
    buf = io.BytesIO()
    s3.download_fileobj(bucket, key, buf)
    buf.seek(0)
    return pd.read_parquet(buf, **kwargs)

def s3_write_parquet(df, bucket, key, compression='snappy'):
    s3 = boto3.client('s3')
    buf = io.BytesIO()
    df.to_parquet(buf, index=False, compression=compression)
    buf.seek(0)
    s3.upload_fileobj(buf, bucket, key)

def s3_list_files(bucket, prefix='', suffix='.parquet'):
    s3 = boto3.client('s3')
    paginator = s3.get_paginator('list_objects_v2')
    return [
        obj['Key']
        for page in paginator.paginate(Bucket=bucket, Prefix=prefix)
        for obj in page.get('Contents', [])
        if obj['Key'].endswith(suffix)
    ]